### Consultas Pyspark – Habilidades de manipulação de dados com Pyspark.
#### Vamos utilizar Pyspark nas consultas das tabelas Deltas na camada Gold.

In [0]:
# Importar bibliotecas
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Configuração inicial da SparkSession com configurações otimizadas
spark = SparkSession.builder \
    .appName("Load Data Silver") \
    .config("spark.sql.shuffle.partitions", "200")  \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

In [0]:
# Caminho base dos dados
PATH_BASE = "dbfs:/mnt/panex/lhdw/gold/"

### Função para limpeza de Dataframes

In [0]:
import inspect
from pyspark.sql import DataFrame

def limpeza_dataframes():
    # Obter todas as variáveis locais
    local_vars = inspect.currentframe().f_back.f_locals
    
    # Coletar os nomes dos dataframes em uma lista separada
    df_names = [var_name for var_name, var_value in local_vars.items() if isinstance(var_value, DataFrame)]
    
    # Iterar sobre a lista de nomes de dataframes e eliminá-los
    for var_name in df_names:
        var_value = local_vars[var_name]
        var_value.unpersist()
        del local_vars[var_name]
        print(f"✅ DataFrame '{var_name}' eliminado e deletado!")


### 🏆 Consulta 1 - Número total de registros de Vendas

In [0]:
# Leitura das tabelas Delta
df_fato = spark.read.format("delta").load(f"{PATH_BASE}fato/")
#Calculo
total_linhas = df_fato.count()
print(f"➡️ Total de linhas na tabela fato: {total_linhas}")

#Limpeza Dataframes
limpeza_dataframes()

➡️ Total de linhas na tabela fato: 965446
✅ DataFrame 'df_fato' eliminado e deletado!


### 🏆 Consulta 2 - Total de Vendas em Fev/2018 por Categoria

In [0]:
# Parametros para otimizar leitura do dados (predicate pushdown)
P_ANO = '2018'
P_MES = '02'
# Leitura das tabelas Delta otimizando com predicate pushdown
df_fato = spark.read.format("delta").load(f"{PATH_BASE}fato/Ano={P_ANO}/Mes={P_MES}/")

df_produtos = spark.read.format("delta").load(f"{PATH_BASE}dimensao/produtos/").filter(F.col("ativo") == True)
df_categorias = spark.read.format("delta").load(f"{PATH_BASE}dimensao/categorias/").filter(F.col("ativo") == True)

df_resultado = (
    df_fato
    .join(F.broadcast(df_produtos), "sk_produtos")
    .join(F.broadcast(df_categorias), "sk_categorias")
    .groupBy("NomeCategoria")
     .agg(F.format_number(F.sum("PrecoTotal"), 2).alias("Total_Vendas"))  # Formata com separação de milhar
    .orderBy(F.desc("Total_Vendas"))
)

# Substituir ponto decimal por vírgula para exibição no formato brasileiro
df_resultado = df_resultado.withColumn("Total_Vendas", F.regexp_replace("Total_Vendas", "\\,", ","))
display(df_resultado)

#Limpeza Dataframes
limpeza_dataframes()

NomeCategoria,Total_Vendas
Confections,"27,442,781.23"
Meat,"24,224,305.48"
Poultry,"21,735,811.50"
Cereals,"20,718,752.67"
Snails,"18,479,933.61"
Beverages,"18,100,216.20"
Produce,"17,968,546.70"
Dairy,"17,367,537.48"
Seafood,"16,225,837.75"
Grain,"16,044,142.81"


✅ DataFrame 'df_fato' eliminado e deletado!
✅ DataFrame 'df_produtos' eliminado e deletado!
✅ DataFrame 'df_categorias' eliminado e deletado!
✅ DataFrame 'df_resultado' eliminado e deletado!


### 🏆 Consulta 3 - Total de vendas de Mar/2018 por Vendedor

In [0]:
# Parametros para otimizar leitura do dados (predicate pushdown)
P_ANO = '2018'
P_MES = '03'
# Leitura das tabelas Delta otimizando com predicate pushdown
df_fato = spark.read.format("delta").load(f"{PATH_BASE}fato/Ano={P_ANO}/Mes={P_MES}/")
df_vendedores = spark.read.format("delta").load(f"{PATH_BASE}dimensao/vendedores/").filter(F.col("ativo") == True)

df_resultado = (
    df_fato
    .join(F.broadcast(df_vendedores), "sk_vendedores")
    .groupBy("Nome")
    .agg(F.format_number(F.sum("Quantidade"), 0).alias("Quantidade"))  # Formata com separação de milhar
    .orderBy(F.desc("Quantidade"))
)

display(df_resultado)

#Limpeza Dataframes
limpeza_dataframes()


Nome,Quantidade
Desiree L Stuart,"185,470"
Lindsay M Chen,"184,629"
Kari D Finley,"184,093"
Bernard L Moody,"183,670"
Julie E Dyer,"183,104"
Christine W Palmer,"182,850"
Sonya E Dickson,"181,926"
Pablo Y Cline,"181,887"
Janet K Flowers,"181,884"
Tonia O Mc Millan,"181,832"


✅ DataFrame 'df_fato' eliminado e deletado!
✅ DataFrame 'df_vendedores' eliminado e deletado!
✅ DataFrame 'df_resultado' eliminado e deletado!


### 🏆 Consulta 4 - Total de vendas em R$ em Fev/2018 por País

In [0]:
# Parametros para otimizar leitura do dados (predicate pushdown)
P_ANO = '2018'
P_MES = '02'
# Leitura das tabelas Delta otimizando com predicate pushdown
df_fato = spark.read.format("delta").load(f"{PATH_BASE}fato/Ano={P_ANO}/Mes={P_MES}/")
df_clientes = spark.read.format("delta").load(f"{PATH_BASE}dimensao/clientes/").filter(F.col("ativo") == True)
df_cidades = spark.read.format("delta").load(f"{PATH_BASE}dimensao/cidades/").filter(F.col("ativo") == True)
df_paises = spark.read.format("delta").load(f"{PATH_BASE}dimensao/paises/").filter(F.col("ativo") == True)

df_resultado = (
    df_fato
    .join(F.broadcast(df_clientes), "sk_clientes")
    .join(F.broadcast(df_cidades), "sk_cidades")
    .join(F.broadcast(df_paises), "sk_paises")
    .groupBy("PaisNome")
    .agg(F.format_number(F.sum("PrecoTotal"), 2).alias("Total_Vendas"))  # Formata com separação de milhar
    .orderBy(F.desc("Total_Vendas"))
)

# Substituir ponto decimal por vírgula para exibição no formato brasileiro
df_resultado = df_resultado.withColumn("Total_Vendas", F.regexp_replace("Total_Vendas", "\\,", ","))

display(df_resultado)

#Limpeza Dataframes
limpeza_dataframes()

PaisNome,Total_Vendas
United States,"212,987,855.56"


✅ DataFrame 'df_fato' eliminado e deletado!
✅ DataFrame 'df_clientes' eliminado e deletado!
✅ DataFrame 'df_cidades' eliminado e deletado!
✅ DataFrame 'df_paises' eliminado e deletado!
✅ DataFrame 'df_resultado' eliminado e deletado!


### 🏆 Consulta 5 - Total de Vendas (R$) mês a mês em 2018

In [0]:
# Leitura das tabelas Delta otimizando
df_fato = spark.read.format("delta").load(f"{PATH_BASE}fato")

df_resultado = (
    df_fato.groupBy("Mes")
    .agg(F.format_number(F.sum("PrecoTotal"), 2).alias("Total_Vendas"))
    .orderBy("Mes")
)

display(df_resultado)

#Limpeza Dataframes
limpeza_dataframes()


Mes,Total_Vendas
1,"212,848,675.48"
2,"212,987,855.56"
3,"212,567,770.99"


✅ DataFrame 'df_fato' eliminado e deletado!
✅ DataFrame 'df_resultado' eliminado e deletado!


### 🏆 Consulta 6 - Total de desconto mês a mês em 2018

In [0]:
# Leitura das tabelas Delta otimizando
df_fato = spark.read.format("delta").load(f"{PATH_BASE}fato")

df_resultado = (
    df_fato.groupBy("Mes")
    .agg(F.format_number(F.sum("Desconto"), 2).alias("Total_Desconto"))
    .orderBy("Mes")
)

display(df_resultado)

#Limpeza Dataframes
limpeza_dataframes()


Mes,Total_Desconto
1,"9,556.50"
2,"9,668.20"
3,"9,636.70"


✅ DataFrame 'df_fato' eliminado e deletado!
✅ DataFrame 'df_resultado' eliminado e deletado!


### 🏆 Consulta 7 - Variação % (MoM) de Total de vendas de Mar/2018 para Fev/2018

In [0]:
# Parametros para otimizar leitura do dados (predicate pushdown)
P_ANO = '2018'
P_MES_ANTES_CONTEXTO = '02'
P_MES_CONTEXTO ='03'
# Leitura das tabelas Delta otimizando com predicate pushdown
df_fato_contexto = spark.read.format("delta").load(f"{PATH_BASE}fato/Ano={P_ANO}/Mes={P_MES_CONTEXTO}/")
df_fato_antes_contexto = spark.read.format("delta").load(f"{PATH_BASE}fato/Ano={P_ANO}/Mes={P_MES_ANTES_CONTEXTO}/")

# Calcular Total_Vendas para cada mês
df_total_vendas_contexto = df_fato_contexto.agg(F.sum("PrecoTotal").alias("Total_Vendas_Atual"))
df_total_vendas_antes_contexto = df_fato_antes_contexto.agg(F.sum("PrecoTotal").alias("Total_Vendas_Anterior"))

# Calcular variação MoM
df_variacao_mom = df_total_vendas_contexto.crossJoin(df_total_vendas_antes_contexto).withColumn(
    "Variacao_MoM", 
    F.format_number((F.col("Total_Vendas_Atual") - F.col("Total_Vendas_Anterior")) / F.col("Total_Vendas_Anterior") * 100, 2)
)

display(df_variacao_mom)

#Limpeza Dataframes
limpeza_dataframes()

Total_Vendas_Atual,Total_Vendas_Anterior,Variacao_MoM
2.1256777098999727E8,2.1298785555999827E8,-0.20


✅ DataFrame 'df_fato_contexto' eliminado e deletado!
✅ DataFrame 'df_fato_antes_contexto' eliminado e deletado!
✅ DataFrame 'df_total_vendas_contexto' eliminado e deletado!
✅ DataFrame 'df_total_vendas_antes_contexto' eliminado e deletado!
✅ DataFrame 'df_variacao_mom' eliminado e deletado!


### 🏆 Consulta 8 - Top 10 produtos com maior Valor de vendas (R$) em Fev/2018

In [0]:
# Parametros para otimizar leitura do dados (predicate pushdown)
P_ANO = '2018'
P_MES = '02'
# Leitura das tabelas Delta otimizando com predicate pushdown
df_fato = spark.read.format("delta").load(f"{PATH_BASE}fato/Ano={P_ANO}/Mes={P_MES}/")
df_produtos = spark.read.format("delta").load(f"{PATH_BASE}dimensao/produtos/").filter(F.col("ativo") == True)

df_resultado = (
    df_fato
    .join(F.broadcast(df_produtos), "sk_produtos")
    .groupBy("ProdutoNome")
    .agg(F.sum("PrecoTotal").alias("Total_Vendas"))
    .orderBy(F.desc("Total_Vendas"))
    .limit(10)
    .withColumn("Total_Vendas", F.format_number(F.col("Total_Vendas"), 2))
)

display(df_resultado)

#Limpeza Dataframes
limpeza_dataframes()

ProdutoNome,Total_Vendas
Tia Maria,"1,022,335.92"
Puree - Passion Fruit,"994,266.88"
Bread - Multigrain,"960,641.76"
Beer - Rickards Red,"929,162.89"
Pork - Hock And Feet Attached,"921,490.44"
Soup Knorr Chili With Beans,"910,777.48"
Vanilla Beans,"906,205.77"
Bread - Calabrese Baguette,"905,010.13"
Beef - Inside Round,"899,911.40"
Tuna - Salad Premix,"897,456.16"


✅ DataFrame 'df_fato' eliminado e deletado!
✅ DataFrame 'df_produtos' eliminado e deletado!
✅ DataFrame 'df_resultado' eliminado e deletado!


### 🏆 Consulta 9 - % de Total de vendas (R$), por Categoria em Fev/2018

In [0]:
P_ANO = '2018'
P_MES = '02'
# Leitura das tabelas Delta otimizando com predicate pushdown
df_fato = spark.read.format("delta").load(f"{PATH_BASE}fato/Ano={P_ANO}/Mes={P_MES}/")
df_produtos = spark.read.format("delta").load(f"{PATH_BASE}dimensao/produtos/").filter(F.col("ativo") == True)
df_categorias = spark.read.format("delta").load(f"{PATH_BASE}dimensao/categorias/").filter(F.col("ativo") == True)

df_total = (
    df_fato
    .agg(F.sum("PrecoTotal").alias("Total_Vendas_Geral"))
)

df_categorias_fev = (
    df_fato
    .join(F.broadcast(df_produtos), "sk_produtos")
    .join(F.broadcast(df_categorias), "sk_categorias")
    .groupBy("NomeCategoria")
    .agg(F.sum("PrecoTotal").alias("Total_Vendas_Categoria"))
)

df_resultado = (
    df_categorias_fev.crossJoin(df_total)
    .withColumn("Share_Percentual", (F.col("Total_Vendas_Categoria") / F.col("Total_Vendas_Geral")) * 100)
    .orderBy(F.desc("Share_Percentual"))
    .withColumn("Share_Percentual", F.format_string("%.2f%%", F.col("Share_Percentual")))
)

display(df_resultado)

#Limpeza Dataframes
limpeza_dataframes()

NomeCategoria,Total_Vendas_Categoria,Total_Vendas_Geral,Share_Percentual
Confections,2.7442781230000168E7,2.1298785555999827E8,12.88%
Meat,2.4224305480000097E7,2.1298785555999827E8,11.37%
Poultry,2.1735811500000127E7,2.1298785555999827E8,10.21%
Cereals,2.0718752669999912E7,2.1298785555999827E8,9.73%
Snails,1.8479933609999962E7,2.1298785555999827E8,8.68%
Beverages,1.810021619999992E7,2.1298785555999827E8,8.50%
Produce,1.796854670000005E7,2.1298785555999827E8,8.44%
Dairy,1.736753747999998E7,2.1298785555999827E8,8.15%
Seafood,1.6225837750000013E7,2.1298785555999827E8,7.62%
Grain,1.604414280999992E7,2.1298785555999827E8,7.53%


✅ DataFrame 'df_fato' eliminado e deletado!
✅ DataFrame 'df_produtos' eliminado e deletado!
✅ DataFrame 'df_categorias' eliminado e deletado!
✅ DataFrame 'df_total' eliminado e deletado!
✅ DataFrame 'df_categorias_fev' eliminado e deletado!
✅ DataFrame 'df_resultado' eliminado e deletado!


### 🏆 Consulta 10 - Ticket médio de Total de vendas por categoria em Jan/2018

In [0]:
P_ANO = '2018'
P_MES = '01'
# Leitura das tabelas Delta otimizando com predicate pushdown
df_fato = spark.read.format("delta").load(f"{PATH_BASE}fato/Ano={P_ANO}/Mes={P_MES}/")
df_produtos = spark.read.format("delta").load(f"{PATH_BASE}dimensao/produtos/").filter(F.col("ativo") == True)
df_categorias = spark.read.format("delta").load(f"{PATH_BASE}dimensao/categorias/").filter(F.col("ativo") == True)

df_resultado = (
    df_fato
    .join(F.broadcast(df_produtos), "sk_produtos")
    .join(F.broadcast(df_categorias), "sk_categorias")
    .groupBy("NomeCategoria")
    .agg(F.avg("PrecoTotal").alias("Ticket_Medio"))
    .orderBy(F.desc("Ticket_Medio"))
    .withColumn("Ticket_Medio", F.format_number(F.col("Ticket_Medio"), 2))
)

display(df_resultado)

#Limpeza Dataframes
limpeza_dataframes()

NomeCategoria,Ticket_Medio
Grain,798.41
Snails,695.48
Dairy,694.39
Meat,684.34
Confections,677.80
Beverages,665.99
Cereals,655.24
Poultry,641.08
Seafood,635.96
Produce,587.16


✅ DataFrame 'df_fato' eliminado e deletado!
✅ DataFrame 'df_produtos' eliminado e deletado!
✅ DataFrame 'df_categorias' eliminado e deletado!
✅ DataFrame 'df_resultado' eliminado e deletado!
